# Wallet Group Positions

Classifies every wallet that traded a single condition into behavioural types,
then plots their **net Yes-equivalent position** over time per group.

### Net Yes-equivalent position
Each trade row contributes a signed delta to the wallet's Yes-equivalent position:

| Action | Delta |
|---|---|
| BUY Yes  | +quantity |
| SELL Yes | −quantity |
| BUY No   | −quantity (buying No = shorting Yes) |
| SELL No  | +quantity (closing a No position) |

Both legs of each trade are present in the data, so volumes are halved.

### Classification heuristics (applied in priority order)

| Group | Criteria |
|---|---|
| **Market Maker** | Net-yes buy ratio 0.35–0.65 across all trades (Yes+No), ≥ `MM_MIN_TRADES` |
| **Whale** | Total USDC (both legs / 2) ≥ `WHALE_USDC_THRESHOLD` |
| **Active Trader** | ≥ `ACTIVE_TRADE_THRESHOLD` trades, not MM or Whale |
| **Retail** | Everyone else |

All classification thresholds are in the **Configuration** cell.

Requires: `pandas`, `pyarrow`, `plotly`

In [319]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Configuration

In [320]:
# ── Data ──────────────────────────────────────────────────────────────────────
ENRICHED_TRADES_DIR = Path("../data/trades_polygon_enriched")
CONDITION_ID        = "0xd6b79f9d36c0ad17befcfb54b7f5c530e0c0a23040b2a1509159c6e09da8c517"

# ── Classification thresholds ─────────────────────────────────────────────────
# Market Maker: two-sided, buy ratio within [MM_BUY_RATIO_LO, MM_BUY_RATIO_HI],
#               and at least MM_MIN_TRADES trades
MM_BUY_RATIO_LO = 0.35
MM_BUY_RATIO_HI = 0.65
MM_MIN_TRADES   = 40

# Whale: total USDC traded (all legs combined) exceeds this value
WHALE_USDC_THRESHOLD = 50_000

# Active Trader: at least this many trades (and not MM / Whale)
ACTIVE_TRADE_THRESHOLD = 20

# ── Position timeline ─────────────────────────────────────────────────────────
# Resample frequency for the group position chart
POSITION_FREQ = "1h"

# ── Watched wallets ──────────────────────────────────────────────────────────
# Set inline, or leave as None to load from the stage-1 workspace parquet.
WATCHED_WALLETS: list[str] | None = None

# Path to stage-1 selected_wallets parquet (used when WATCHED_WALLETS is None)
WATCHED_WALLETS_PATH = Path("../data/trade_signals_workspace_v2/selected_wallets_v2.parquet")

# ── Colours per group ─────────────────────────────────────────────────────────
GROUP_COLORS = {
    "Market Maker":   "#1f77b4",
    "Whale":          "#9467bd",
    "Active Trader":  "#ff7f0e",
    "Retail":         "#2ca02c",
}


## 1 · Load trades

In [321]:
shard_char   = CONDITION_ID[2].lower()
parquet_file = ENRICHED_TRADES_DIR / f"enriched_{shard_char}.parquet"
print(f"Reading shard: {parquet_file}")

COLS = ["condition_id", "ts", "outcome", "side",
        "price", "quantity", "usdc_amount", "position", "wallet",
        "question", "avail_copy_count"]

raw = pd.read_parquet(parquet_file, columns=COLS)
raw = raw[raw["condition_id"] == CONDITION_ID].copy()
raw["ts"] = pd.to_datetime(raw["ts"], utc=True)
raw.sort_values("ts", inplace=True)
raw.reset_index(drop=True, inplace=True)

question = raw["question"].iloc[0]
print(f"Question : {question}")
print(f"Rows     : {len(raw):,}   Wallets: {raw['wallet'].nunique():,}")
print(f"Period   : {raw['ts'].min()}  →  {raw['ts'].max()}")

Reading shard: ../data/trades_polygon_enriched/enriched_d.parquet
Question : Will Trump's remarks not air?
Rows     : 1,499   Wallets: 218
Period   : 2026-03-30 22:58:33+00:00  →  2026-04-02 08:57:01+00:00


## 1a · Load watched wallets

In [322]:
if WATCHED_WALLETS:
    watched_set = set(w.lower() for w in WATCHED_WALLETS)
    print(f"Using {len(watched_set)} manually specified watched wallets.")
elif WATCHED_WALLETS_PATH.exists():
    _wdf = pd.read_parquet(WATCHED_WALLETS_PATH, columns=["wallet"])
    watched_set = set(_wdf["wallet"].str.lower().tolist())
    print(f"Loaded {len(watched_set)} wallets from {WATCHED_WALLETS_PATH.name}")
else:
    watched_set = set()
    print("No watched wallets configured — skipping watched-wallet overlay.")

# Keep only wallets that actually traded this condition
all_wallets_in_condition = set(raw["wallet"].str.lower().unique())
watched_set = watched_set & all_wallets_in_condition
print(f"Watched wallets active in this condition: {len(watched_set):,}")

Loaded 117 wallets from selected_wallets_v2.parquet
Watched wallets active in this condition: 10


## 2 · Classify wallets

In [323]:
# Use all rows (both Yes and No legs) for classification stats.
# Volume is divided by 2 because each matched trade produces two rows.

# Net-Yes-bullish: trades that increase Yes exposure
#   BUY Yes  -> bullish  |  SELL No -> bullish
#   SELL Yes -> bearish  |  BUY No  -> bearish
raw["is_net_yes_bullish"] = (
    ((raw["outcome"] == "Yes") & (raw["side"] == "BUY"))
    | ((raw["outcome"] == "No")  & (raw["side"] == "SELL"))
)

wallet_stats = raw.groupby("wallet").agg(
    trade_count       = ("quantity",             "count"),
    total_usdc        = ("usdc_amount",           "sum"),
    n_buys            = ("side", lambda s: (s == "BUY").sum()),
    n_sells           = ("side", lambda s: (s == "SELL").sum()),
    n_net_yes_bullish = ("is_net_yes_bullish",    "sum"),
    first_trade       = ("ts",                    "min"),
    last_trade        = ("ts",                    "max"),
    avg_copy_cnt      = ("avail_copy_count",      "mean"),
).copy()


# Net-Yes-bullish buy_ratio: fraction of all trades that increase Yes exposure
wallet_stats["buy_ratio"]   = wallet_stats["n_net_yes_bullish"] / wallet_stats["trade_count"]
wallet_stats["two_sided"]   = (wallet_stats["buy_ratio"] > 0) & (wallet_stats["buy_ratio"] < 1)
wallet_stats["active_days"] = (
    (wallet_stats["last_trade"] - wallet_stats["first_trade"]).dt.total_seconds() / 86_400
)


def classify(row) -> str:
    """Assign a single wallet to one behavioural group.

    Priority: Market Maker > Whale > Active Trader > Retail
    """
    is_mm = (
        row["two_sided"]
        and (MM_BUY_RATIO_LO <= row["buy_ratio"] <= MM_BUY_RATIO_HI)
        and (row["trade_count"] >= MM_MIN_TRADES)
    )
    if is_mm:
        return "Market Maker"
    if row["total_usdc"] >= WHALE_USDC_THRESHOLD:
        return "Whale"
    if row["trade_count"] >= ACTIVE_TRADE_THRESHOLD:
        return "Active Trader"
    return "Retail"


wallet_stats["group"] = wallet_stats.apply(classify, axis=1)

group_counts = wallet_stats["group"].value_counts()
print("Wallet counts per group:")
for g, n in group_counts.items():
    total_vol = wallet_stats.loc[wallet_stats["group"] == g, "total_usdc"].sum()
    print(f"  {g:<15} {n:>5} wallets   ${total_vol:>14,.0f} USDC volume")

wallet_stats.head(5)


Wallet counts per group:
  Retail            205 wallets   $        21,484 USDC volume
  Active Trader       7 wallets   $         5,263 USDC volume
  Market Maker        6 wallets   $         3,592 USDC volume


,trade_count,total_usdc,n_buys,n_sells,n_net_yes_bullish,first_trade,last_trade,avg_copy_cnt,buy_ratio,two_sided,active_days,group
wallet,,,,,,,,,,,,
0x0061f603a457754bae0241f1fb34f0ce6a5e8fd6,1,0.200000,1,0,0,2026-04-02 06:04:05+00:00,2026-04-02 06:04:05+00:00,0.000000,0.000000,False,0.000000,Retail
0x007ea4cbbdf4d230e12e3913d706b796ede9c290,6,5.636560,4,2,2,2026-04-01 18:33:29+00:00,2026-04-01 18:57:59+00:00,5.666667,0.333333,True,0.017014,Retail
0x00c091b287196c418a62b936905eb1a9b8da4314,2,6.365439,1,1,1,2026-04-01 09:18:21+00:00,2026-04-01 14:07:01+00:00,0.000000,0.500000,True,0.200463,Retail
0x017bbd948248be2ae58d8fbfad186428c7291ef4,7,26.434928,2,5,6,2026-04-01 03:57:19+00:00,2026-04-01 09:28:21+00:00,0.142857,0.857143,True,0.229884,Retail
0x01b0c2967a56de35210beffe512cd7d4ae805aaa,8,47.879999,8,0,0,2026-04-02 02:29:23+00:00,2026-04-02 02:29:23+00:00,2.500000,0.000000,False,0.000000,Retail


## 2b · Classify watched wallets

In [324]:
if watched_set:
    watched_raw = raw[raw["wallet"].str.lower().isin(watched_set)].copy()

    watched_raw["is_net_yes_bullish"] = (
        ((watched_raw["outcome"] == "Yes") & (watched_raw["side"] == "BUY"))
        | ((watched_raw["outcome"] == "No")  & (watched_raw["side"] == "SELL"))
    )

    watched_stats = watched_raw.groupby("wallet").agg(
        trade_count       = ("quantity",             "count"),
        total_usdc        = ("usdc_amount",           "sum"),
        n_buys            = ("side", lambda s: (s == "BUY").sum()),
        n_sells           = ("side", lambda s: (s == "SELL").sum()),
        n_net_yes_bullish = ("is_net_yes_bullish",    "sum"),
        first_trade       = ("ts",                    "min"),
        last_trade        = ("ts",                    "max"),
        avg_copy_cnt      = ("avail_copy_count",      "mean"),
    ).copy()

    watched_stats["buy_ratio"]   = watched_stats["n_net_yes_bullish"] / watched_stats["trade_count"]
    watched_stats["two_sided"]   = (watched_stats["buy_ratio"] > 0) & (watched_stats["buy_ratio"] < 1)
    watched_stats["active_days"] = (
        (watched_stats["last_trade"] - watched_stats["first_trade"]).dt.total_seconds() / 86_400
    )
    watched_stats["group"] = watched_stats.apply(classify, axis=1)

    print("Watched wallet counts per group:")
    for g, n in watched_stats["group"].value_counts().items():
        vol = watched_stats.loc[watched_stats["group"] == g, "total_usdc"].sum()
        print(f"  {g:<15} {n:>5} wallets   ${vol:>14,.0f} USDC volume")
    display(watched_stats.head(5))
else:
    watched_stats = pd.DataFrame()
    print("No watched wallets — skipping.")

Watched wallet counts per group:
  Retail             10 wallets   $         2,183 USDC volume


,trade_count,total_usdc,n_buys,n_sells,n_net_yes_bullish,first_trade,last_trade,avg_copy_cnt,buy_ratio,two_sided,active_days,group
wallet,,,,,,,,,,,,
0x0cb10c40b0776e9ee8cef970af85724654dda76c,2,130.878000,1,1,1,2026-04-01 03:10:09+00:00,2026-04-01 14:02:59+00:00,0.500000,0.500000,True,0.453356,Retail
0x134a63b764ac7b008356e8db1857db94e6b09e42,4,209.203380,4,0,4,2026-04-01 22:23:47+00:00,2026-04-01 22:23:47+00:00,1.000000,1.000000,False,0.000000,Retail
0x355e5ae20cc1a3a2164f818e4bac8d22dd72a038,19,737.235142,14,5,14,2026-04-01 16:36:45+00:00,2026-04-01 20:50:45+00:00,2.368421,0.736842,True,0.176389,Retail
0x40c2348351024cce5ba6fccd6015b77bbb77b2fc,4,10.170000,4,0,4,2026-04-01 05:41:25+00:00,2026-04-01 21:30:27+00:00,3.000000,1.000000,False,0.659051,Retail
0x4247c89f4a9236a6b5fb24502023bcc98ab1455f,1,1.834000,1,0,0,2026-04-01 12:10:11+00:00,2026-04-01 12:10:11+00:00,0.000000,0.000000,False,0.000000,Retail


### 2a · Wallet stats summary per group

In [325]:
summary = (
    wallet_stats.groupby("group")[["trade_count", "total_usdc", "buy_ratio", "active_days"]]
    .agg(["median", "mean", "max"])
    .round(2)
)
summary

trade_count            total_usdc                  buy_ratio  \
                   median   mean max     median    mean      max    median   
group                                                                        
Active Trader        30.0  31.14  38     641.29  751.87  2733.99      0.45   
Market Maker         57.5  61.33  97     204.32  598.75  2475.90      0.46   
Retail                3.0   4.45  19      12.43  104.80  2448.39      0.50   

                          active_days              
               mean   max      median  mean   max  
group                                              
Active Trader  0.44  0.61        0.94  0.97  2.30  
Market Maker   0.48  0.65        0.60  0.56  0.81  
Retail         0.50  1.00        0.02  0.19  2.30

## 3 · Build position timeline per group

In [326]:
# Join group label onto every trade row (all outcomes)
all_g = raw.join(wallet_stats[["group"]], on="wallet")

# Signed Yes-equivalent delta per trade row:
#   BUY  Yes  → +quantity  (acquiring Yes shares)
#   SELL Yes  → -quantity  (disposing Yes shares)
#   BUY  No   → -quantity  (buying No = shorting Yes)
#   SELL No   → +quantity  (closing a No position)
sign = (
    all_g["side"].map({"BUY": 1, "SELL": -1})
    * all_g["outcome"].map({"Yes": 1, "No": -1})
)
all_g["pos_delta"] = all_g["quantity"] * sign



def group_position_timeline(
    all_g: pd.DataFrame,
    freq: str,
) -> pd.DataFrame:
    """
    Return a DataFrame indexed by `freq`-resampled timestamps with one column
    per group, containing the cumulative net Yes-position of that group at
    each point in time.
    """
    groups    = list(GROUP_COLORS.keys())
    ts_index  = pd.date_range(
        all_g["ts"].min().floor(freq),
        all_g["ts"].max().ceil(freq),
        freq=freq,
        tz="UTC",
    )

    result = pd.DataFrame(index=ts_index)
    result.index.name = "ts"

    for group in groups:
        g_rows = all_g[all_g["group"] == group].set_index("ts")
        # Sum of deltas within each bucket
        period_delta = g_rows["pos_delta"].resample(freq).sum().reindex(ts_index, fill_value=0)
        # Cumulative sum → running position
        result[group] = period_delta.cumsum().values

    return result


pos_timeline = group_position_timeline(all_g, POSITION_FREQ)
print(f"Timeline shape: {pos_timeline.shape}  (freq={POSITION_FREQ})")

# ── Watched wallet position timeline (same logic, filtered to watched_set) ────
if not watched_stats.empty:
    watched_g = raw[raw["wallet"].str.lower().isin(watched_set)].copy()
    watched_g = watched_g.join(watched_stats[["group"]], on="wallet")
    sign_w = (
        watched_g["side"].map({"BUY": 1, "SELL": -1})
        * watched_g["outcome"].map({"Yes": 1, "No": -1})
    )
    watched_g["pos_delta"] = watched_g["quantity"] * sign_w
    watched_timeline = group_position_timeline(watched_g, POSITION_FREQ)
    print(f"Watched timeline shape: {watched_timeline.shape}")
else:
    watched_timeline = pd.DataFrame()

# ── VWAP per bucket (Yes-leg only; No prices flipped to Yes-equivalent) ─────
yes_rows = raw[raw["outcome"] == "Yes"].copy()
no_rows  = raw[raw["outcome"] == "No"].copy()
no_rows["price"] = 1 - no_rows["price"]
yes_equiv = pd.concat([yes_rows, no_rows], ignore_index=True)
yes_equiv["pq"] = yes_equiv["price"] * yes_equiv["quantity"]
yes_equiv = yes_equiv.set_index("ts").sort_index()

vwap = (
    yes_equiv.resample(POSITION_FREQ)["pq", "quantity"]
    .sum()
    .reindex(pos_timeline.index)
)
vwap["vwap"] = vwap["pq"] / vwap["quantity"]
# Forward-fill gaps (buckets with no trades keep last known price)
vwap["vwap"] = vwap["vwap"].ffill()

pos_timeline.tail(4)


Timeline shape: (60, 4)  (freq=1h)
Watched timeline shape: (30, 4)


,Market Maker,Whale,Active Trader,Retail
ts,,,,
2026-04-02 06:00:00+00:00,707.532511,0.0,-2731.090608,2023.558097
2026-04-02 07:00:00+00:00,707.532511,0.0,-2731.090608,2023.558097
2026-04-02 08:00:00+00:00,707.532511,0.0,-2731.090608,2023.558097
2026-04-02 09:00:00+00:00,707.532511,0.0,-2731.090608,2023.558097


## 4 · Plot

In [327]:
def hex_to_rgba(hex_color: str, alpha: float = 0.15) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def plot_wallet_groups(
    pos_timeline:     pd.DataFrame,
    wallet_stats:     pd.DataFrame,
    vwap:             pd.DataFrame,
    title:            str,
    freq:             str,
    watched_timeline: pd.DataFrame | None = None,
    watched_stats:    pd.DataFrame | None = None,
) -> go.Figure:
    """
    Two-row figure:
      Row 1 – Lines: net Yes-position per wallet group + VWAP on secondary y-axis.
               If watched_timeline is provided, overlays dashed lines per group
               for the watched-wallet subset.
      Row 2 – Signed bars: net Yes-volume per group per bucket (trade count in tooltip)
    """
    # ── Per-bucket signed volume and trade count per group ───────────────────────
    grp_ts = all_g.set_index("ts").groupby("group")

    signed_vol = (
        grp_ts["pos_delta"]
        .resample(freq)
        .sum()
        .unstack(level=0, fill_value=0)
        .reindex(pos_timeline.index, fill_value=0)
    )

    trade_counts = (
        grp_ts["pos_delta"]
        .resample(freq)
        .count()
        .unstack(level=0, fill_value=0)
        .reindex(pos_timeline.index, fill_value=0)
    )

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.06,
        subplot_titles=("Cumulative net Yes-position by wallet group",
                        "Signed Yes-volume by wallet group"),
        specs=[[{"secondary_y": True}],
               [{"secondary_y": False}]],
    )

    groups = list(GROUP_COLORS.keys())

    # ── Row 1: full-population position lines ─────────────────────────────────────
    for group in groups:
        if group not in pos_timeline.columns:
            continue
        color     = GROUP_COLORS[group]
        n_wallets = (wallet_stats["group"] == group).sum()
        vol       = wallet_stats.loc[wallet_stats["group"] == group, "total_usdc"].sum()
        y = pos_timeline[group]

        # Shaded band
        fig.add_trace(
            go.Scatter(
                x=list(pos_timeline.index) + list(pos_timeline.index[::-1]),
                y=list(y) + [0] * len(y),
                fill="toself",
                fillcolor=hex_to_rgba(color, alpha=0.12),
                line=dict(width=0),
                hoverinfo="skip",
                showlegend=False,
                legendgroup=group,
            ),
            row=1, col=1, secondary_y=False,
        )
        # Line
        fig.add_trace(
            go.Scatter(
                x=pos_timeline.index,
                y=y,
                name=f"{group} ({n_wallets} wallets, ${vol:,.0f})",
                mode="lines",
                line=dict(color=color, width=1.5),
                hovertemplate=(
                    f"<b>{group}</b><br>"
                    "%{x|%Y-%m-%d %H:%M UTC}<br>"
                    "Net position: %{y:,.0f} shares<extra></extra>"
                ),
                legendgroup=group,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── Row 1: watched-wallet overlay (dashed, same group colours) ───────────────
    if watched_timeline is not None and not watched_timeline.empty and watched_stats is not None:
        for group in groups:
            if group not in watched_timeline.columns:
                continue
            color = GROUP_COLORS[group]
            n_w   = (watched_stats["group"] == group).sum()
            if n_w == 0:
                continue
            fig.add_trace(
                go.Scatter(
                    x=watched_timeline.index,
                    y=watched_timeline[group],
                    name=f"{group} — watched ({n_w})",
                    mode="lines",
                    line=dict(color=color, width=1.5, dash="dash"),
                    hovertemplate=(
                        f"<b>{group} (watched)</b><br>"
                        "%{x|%Y-%m-%d %H:%M UTC}<br>"
                        "Net position: %{y:,.0f} shares<extra></extra>"
                    ),
                    legendgroup=f"{group}_watched",
                ),
                row=1, col=1, secondary_y=False,
            )

    # ── Row 1 secondary y: VWAP ───────────────────────────────────────────────────
    fig.add_trace(
        go.Scatter(
            x=vwap.index,
            y=vwap["vwap"],
            name="VWAP (Yes price)",
            mode="lines",
            line=dict(color="#333333", width=1.5, dash="dot"),
            hovertemplate=(
                "<b>VWAP</b><br>"
                "%{x|%Y-%m-%d %H:%M UTC}<br>"
                "Price: %{y:.3f}<extra></extra>"
            ),
            legendgroup="vwap",
        ),
        row=1, col=1, secondary_y=True,
    )

    # ── Row 2: signed volume bars ─────────────────────────────────────────────────
    for group in groups:
        if group not in signed_vol.columns:
            continue
        counts     = trade_counts[group] if group in trade_counts.columns else None
        customdata = counts.values.reshape(-1, 1) if counts is not None else None
        fig.add_trace(
            go.Bar(
                x=signed_vol.index,
                y=signed_vol[group],
                name=group,
                marker_color=GROUP_COLORS[group],
                customdata=customdata,
                hovertemplate=(
                    f"<b>{group}</b><br>"
                    "%{x|%Y-%m-%d %H:%M UTC}<br>"
                    "Net volume: %{y:,.0f} shares<br>"
                    "Trades: %{customdata[0]:,}<extra></extra>"
                ),
                legendgroup=group,
                showlegend=False,
            ),
            row=2, col=1,
        )

    fig.update_layout(
        title=title,
        template="plotly_white",
        hovermode="x unified",
        barmode="relative",
        legend=dict(orientation="h", y=1.06),
        margin=dict(t=130),
    )
    fig.update_yaxes(title_text="Net Yes-position (shares)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="VWAP (Yes price)",          row=1, col=1, secondary_y=True,
                     range=[0, 1], tickformat=".2f", showgrid=False)
    fig.update_yaxes(title_text="Net Yes-volume (shares)",   row=2, col=1)
    fig.update_xaxes(title_text="Time (UTC)",                row=2, col=1)

    return fig


title = f"Wallet group positions — {question}<br><sup>{CONDITION_ID}</sup>"
fig = plot_wallet_groups(
    pos_timeline, wallet_stats, vwap, title, POSITION_FREQ,
    watched_timeline=watched_timeline if not watched_timeline.empty else None,
    watched_stats=watched_stats if not watched_stats.empty else None,
)
fig.show(renderer="browser")

## 5 · Classification scatter — explore wallet space

In [328]:
fig2 = go.Figure()

for group in GROUP_COLORS:
    sub = wallet_stats[wallet_stats["group"] == group]
    fig2.add_trace(go.Scatter(
        x=sub["trade_count"],
        y=sub["total_usdc"],
        mode="markers",
        name=group,
        marker=dict(
            color=GROUP_COLORS[group],
            size=6,
            opacity=0.6,
            line=dict(width=0),
        ),
        customdata=sub[["buy_ratio", "active_days"]].values,
        hovertemplate=(
            f"<b>{group}</b><br>"
            "Trades: %{x:,}<br>"
            "USDC vol: $%{y:,.0f}<br>"
            "Buy ratio: %{customdata[0]:.2f}<br>"
            "Active days: %{customdata[1]:.1f}<extra></extra>"
        ),
        legendgroup=group,
    ))

fig2.update_layout(
    title="Wallet classification scatter (trade count vs USDC volume)",
    xaxis=dict(title="Trade count", type="log"),
    yaxis=dict(title="Total USDC traded (Yes leg)", type="log"),
    template="plotly_white",
    hovermode="closest",
    legend=dict(orientation="h", y=1.06),
)
fig2.show(renderer="browser")

## 6 · Watched wallet trades — candles by group

In [329]:
if watched_stats.empty:
    print("No watched wallets — skipping candle chart.")
else:
    # ── Yes-equivalent price on every watched-wallet row ─────────────────────
    wg = watched_g.copy()
    wg["yes_price"] = wg["price"].where(wg["outcome"] == "Yes", 1 - wg["price"])

    active_groups = [g for g in GROUP_COLORS if g in watched_stats["group"].values]

    INC = "#26a69a"
    DEC = "#ef5350"

    # ── Helpers ───────────────────────────────────────────────────────────────
    def build_ohlcv(df: pd.DataFrame, freq: str) -> pd.DataFrame:
        base  = df.set_index("ts").sort_index()
        ohlcv = base["yes_price"].resample(freq).agg(
            open="first", high="max", low="min", close="last"
        )
        buys  = base[base["is_net_yes_bullish"] == True]
        sells = base[base["is_net_yes_bullish"] == False]

        def wavg(rows):
            pq = (rows["yes_price"] * rows["quantity"]).resample(freq).sum()
            q  = rows["quantity"].resample(freq).sum()
            return (pq / q).where(q > 0)

        out = ohlcv.copy()
        out["avg_buy"]   = wavg(buys)
        out["avg_sell"]  = wavg(sells)
        out["volume"]    = base["quantity"].resample(freq).sum()
        out["net_delta"] = base["pos_delta"].resample(freq).sum()
        out["trade_cnt"] = base["pos_delta"].resample(freq).count()
        return out.dropna(subset=["open"])

    # ── Pre-compute ───────────────────────────────────────────────────────────
    datasets = {g: build_ohlcv(wg[wg["group"] == g], POSITION_FREQ) for g in active_groups}

    fig3 = make_subplots(rows=1, cols=1, specs=[[{"secondary_y": True}]])

    for group in active_groups:
        d     = datasets[group]
        color = GROUP_COLORS[group]
        idx   = d.index
        bar_colors = [INC if nd >= 0 else DEC for nd in d["net_delta"]]

        # Candlestick
        fig3.add_trace(go.Candlestick(
            x=idx,
            open=d["open"], high=d["high"], low=d["low"], close=d["close"],
            name=group,
            increasing=dict(line=dict(color=INC), fillcolor=INC),
            decreasing=dict(line=dict(color=DEC), fillcolor=DEC),
            showlegend=True, legendgroup=group,
            hovertext=[
                f"<b>{group}</b>  O:{o:.3f} H:{h:.3f} L:{l:.3f} C:{c:.3f}<br>"
                f"Avg buy: {'—' if np.isnan(ab) else f'{ab:.3f}'}  "
                f"Avg sell: {'—' if np.isnan(as_) else f'{as_:.3f}'}<br>"
                f"Net Δ: {nd:+,.0f}  Vol: {v:,.0f}  Trades: {int(tc):,}"
                for o, h, l, c, ab, as_, nd, v, tc in zip(
                    d["open"], d["high"], d["low"], d["close"],
                    d["avg_buy"].fillna(float("nan")),
                    d["avg_sell"].fillna(float("nan")),
                    d["net_delta"], d["volume"], d["trade_cnt"],
                )
            ],
            hoverinfo="text+x",
        ), row=1, col=1, secondary_y=False)

        # Avg buy markers
        fig3.add_trace(go.Scatter(
            x=idx, y=d["avg_buy"],
            name=f"Avg buy — {group}", mode="markers",
            marker=dict(symbol="triangle-up", color=color, size=8, line=dict(width=1, color="#ffffff")),
            legendgroup=group, showlegend=True,
            hovertemplate=f"<b>{group}</b> avg buy: %{{y:.3f}}<extra></extra>",
        ), row=1, col=1, secondary_y=False)

        # Avg sell markers
        fig3.add_trace(go.Scatter(
            x=idx, y=d["avg_sell"],
            name=f"Avg sell — {group}", mode="markers",
            marker=dict(symbol="triangle-down", color=color, size=8, line=dict(width=1, color="#ffffff")),
            legendgroup=group, showlegend=True,
            hovertemplate=f"<b>{group}</b> avg sell: %{{y:.3f}}<extra></extra>",
        ), row=1, col=1, secondary_y=False)

        # Volume bars (secondary y)
        fig3.add_trace(go.Bar(
            x=idx, y=d["volume"],
            name=f"Volume — {group}",
            marker_color=bar_colors, opacity=0.25,
            legendgroup=group, showlegend=False,
            hoverinfo="skip",
        ), row=1, col=1, secondary_y=True)

    # Market VWAP — always visible
    fig3.add_trace(go.Scatter(
        x=vwap.index, y=vwap["vwap"],
        name="Market VWAP", mode="lines",
        line=dict(color="#333333", width=1.5),
        showlegend=True,
        hovertemplate="Market VWAP: %{y:.3f}<extra></extra>",
    ), row=1, col=1, secondary_y=False)

    fig3.update_layout(
        title=(
            f"Watched wallet trades by group — {question}"
            f"<br><sup>{CONDITION_ID}  ·  {POSITION_FREQ} buckets  ·  "
            f"green = net bullish, red = net bearish</sup>"
        ),
        template="plotly_white",
        hovermode="x unified",
        barmode="overlay",
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.06),
        margin=dict(t=120),
        height=600,
    )
    fig3.update_yaxes(title_text="Yes price", range=[0, 1], secondary_y=False)
    fig3.update_yaxes(title_text="Volume (shares)", showgrid=False, secondary_y=True)
    fig3.update_xaxes(title_text="Time (UTC)")
    fig3.show(renderer="browser")
